In [ ]:
# ============ SoccerNet OSNet ReID — triplet loss, celda única Kaggle (v2) ============
import csv, gc, json, math, os, subprocess, sys, time
from concurrent.futures import ThreadPoolExecutor
from pathlib import Path

# ---------------- CONFIG ----------------
DATA_PATH = None                 # None = autodetecta; ruta manual si adjuntas varios datasets
EPOCHS = 12                      # objetivo total, también al reanudar
P_IDENTITIES = 16                # identidades por batch
K_IMAGES = 4                     # imágenes por identidad; batch efectivo = P * K = 64
IMAGE_SIZE = (256, 128)          # (alto, ancho)
LR = 1e-4
WEIGHT_DECAY = 1e-4
MARGIN = 0.3
CACHE_THREADS = 4                # solo para construir la caché inicial
MAX_HOURS = 11.0                 # guarda last.pt antes de alcanzar el límite Kaggle
SAVE_EVERY_STEPS = 200
OUT = Path('/kaggle/working/out')
SMOKE = False                    # True = 1 epoch, 12 batches para comprobar el pipeline
RESUME = True                    # si encuentra un único lastOsnet.pt adjunto, restaura optimizer y scaler
# ----------------------------------------
if SMOKE:
    EPOCHS = 1

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image
from torch.utils.data import Sampler

assert torch.cuda.is_available(), 'Activa GPU en Settings > Accelerator'
device = torch.device('cuda:0')   # una sola GPU: DataParallel con batch 64 no compensa
print('PyTorch:', torch.__version__, '| CUDA:', torch.version.cuda, '| GPU:', torch.cuda.get_device_name(0))
torch.backends.cudnn.benchmark = True

# ---------- dataset adjunto ----------
if DATA_PATH is not None:
    DATA = Path(DATA_PATH)
else:
    candidates = []
    for summary_path in Path('/kaggle/input').rglob('summary.json'):
        root = summary_path.parent
        if all((root / 'images' / split).is_dir() for split in ('train', 'val', 'test')):
            candidates.append(root)
    candidates = sorted(set(candidates))
    if len(candidates) != 1:
        raise RuntimeError(f'Esperaba un dataset soccernet-reid; encontrados: {candidates}. Define DATA_PATH.')
    DATA = candidates[0]
assert (DATA / 'manifest.csv').is_file(), f'No encuentro manifest.csv en {DATA}'
summary = json.loads((DATA / 'summary.json').read_text(encoding='utf-8'))
print('Dataset:', DATA)
print('Splits:', summary['splits'], '| validación por partido:', summary['validation_game_ids'])

# ---------- Torchreid, sin reinstalar torch/torchvision de Kaggle ----------
TORCHREID_DIR = Path('/kaggle/working/deep-person-reid')
if not TORCHREID_DIR.exists():
    subprocess.check_call([
        'git', 'clone', '--depth', '1',
        'https://github.com/KaiyangZhou/deep-person-reid.git', str(TORCHREID_DIR)
    ])
subprocess.check_call([
    sys.executable, '-m', 'pip', 'install', '-q',
    'gdown', 'yacs', 'tensorboard', 'tensorboardX'
])
sys.path.insert(0, str(TORCHREID_DIR))
from torchreid.models import build_model
print('Torchreid source:', TORCHREID_DIR)

# ---------- caché uint8 en RAM (decodifica + redimensiona una sola vez) ----------
H, W = IMAGE_SIZE

def list_samples(split, fixed_samples=None):
    root = DATA / 'images' / split
    class_dirs = sorted(p for p in root.iterdir() if p.is_dir())
    samples = []
    for label, class_dir in enumerate(class_dirs):
        paths = sorted(class_dir.glob('*.jpg'))
        if fixed_samples is not None and paths:
            rng = np.random.default_rng(2026 + label)
            sel = rng.choice(len(paths), size=min(fixed_samples, len(paths)), replace=False)
            paths = [paths[int(i)] for i in sel]
        samples.extend((p, label) for p in paths)
    if not samples:
        raise RuntimeError(f'Sin recortes en {root}')
    return samples

def load_one(path):
    with Image.open(path) as im:
        return np.asarray(im.convert('RGB').resize((W, H), Image.BILINEAR), dtype=np.uint8)

def build_cache(split, fixed_samples=None):
    samples = list_samples(split, fixed_samples)
    x = np.empty((len(samples), H, W, 3), dtype=np.uint8)
    t0 = time.time()
    with ThreadPoolExecutor(CACHE_THREADS) as ex:
        for i, arr in enumerate(ex.map(load_one, (p for p, _ in samples))):
            x[i] = arr
    labels = np.asarray([l for _, l in samples], dtype=np.int64)
    class_indices = {}
    for i, l in enumerate(labels):
        class_indices.setdefault(int(l), []).append(i)
    print(f'Caché {split}: {len(x):,} crops, {x.nbytes/1e9:.2f} GB, {time.time()-t0:.0f}s', flush=True)
    return x, labels, class_indices

train_x, train_y, train_ci = build_cache('train')
val_x_np, val_y_np, val_ci = build_cache('val', fixed_samples=4)
val_x = torch.from_numpy(val_x_np).to(device)          # uint8 NHWC, ~60 MB
val_y = torch.from_numpy(val_y_np).to(device)
train_y_t = torch.from_numpy(train_y)

MEAN = (0.485, 0.456, 0.406)
STD = (0.229, 0.224, 0.225)
mean_t = torch.tensor(MEAN, device=device).view(1, 3, 1, 1)
std_t = torch.tensor(STD, device=device).view(1, 3, 1, 1)

def prep(x_uint8, train):
    """uint8 NHWC (GPU) -> float normalizado NCHW channels_last, con augmentation opcional en GPU."""
    x = x_uint8.permute(0, 3, 1, 2).float().div_(255)
    if train:
        B = x.shape[0]
        flip = torch.rand(B, device=x.device) < 0.5
        x = torch.where(flip[:, None, None, None], x.flip(3), x)
        def r(a):
            return torch.empty(B, 1, 1, 1, device=x.device).uniform_(1 - a, 1 + a)
        x = x * r(0.15)                                           # brillo
        m = x.mean(dim=(1, 2, 3), keepdim=True)
        x = (x - m) * r(0.15) + m                                 # contraste
        g = 0.299 * x[:, 0:1] + 0.587 * x[:, 1:2] + 0.114 * x[:, 2:3]
        x = ((x - g) * r(0.12) + g).clamp_(0, 1)                  # saturación
    x = (x - mean_t) / std_t
    return x.contiguous(memory_format=torch.channels_last)

class PKBatchSampler(Sampler):
    def __init__(self, class_indices, p, k, steps=None, seed=1234):
        self.class_indices = class_indices
        self.classes = np.asarray(sorted(class_indices))
        self.p, self.k, self.seed = p, k, seed
        self.epoch = 0
        if len(self.classes) < p:
            raise RuntimeError(f'P={p} requiere al menos P identidades; hay {len(self.classes)}')
        self.steps = steps or math.ceil(sum(map(len, class_indices.values())) / (p * k))

    def __len__(self):
        return self.steps

    def __iter__(self):
        rng = np.random.default_rng(self.seed + self.epoch)
        for _ in range(self.steps):
            chosen = rng.choice(self.classes, size=self.p, replace=False)
            batch = []
            for label in chosen:
                idxs = self.class_indices[int(label)]
                batch.extend(rng.choice(idxs, size=self.k, replace=len(idxs) < self.k).tolist())
            yield batch

    def set_epoch(self, epoch):
        self.epoch = epoch

P = min(P_IDENTITIES, len(train_ci))
K = K_IMAGES
train_sampler = PKBatchSampler(train_ci, P, K, steps=12 if SMOKE else None)
val_P = min(P, len(val_ci))
val_sampler = PKBatchSampler(val_ci, val_P, K, steps=min(32, math.ceil(len(val_ci) / val_P)), seed=987)
print(f'train={len(train_y):,} crops / {len(train_ci)} IDs | val={len(val_y):,} crops / {len(val_ci)} IDs | '
      f'batch={P}x{K}={P*K} | {len(train_sampler)} pasos/epoch')

# ---------- modelo y checkpoints ----------
NUM_CLASSES = len(train_ci)
model = build_model(
    name='osnet_x1_0', num_classes=NUM_CLASSES, loss='triplet',
    pretrained=True, use_gpu=True
).to(device).to(memory_format=torch.channels_last)
print('OSNet x1.0 ImageNet-pretrained; fine-tuning triplet en SoccerNet')
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS)
scaler = torch.amp.GradScaler('cuda')
OUT.mkdir(parents=True, exist_ok=True)
metrics_path = OUT / 'metrics.csv'
if not metrics_path.exists():
    metrics_path.write_text('epoch,train_loss,val_loss,val_rank1,val_map,minutes\n', encoding='utf-8')

def checkpoint_pack(epoch, best_map, val_rank1, val_loss):
    return {
        'state_dict': model.state_dict(),
        'epoch': epoch,
        'best_val_map': best_map,
        'val_rank1': val_rank1,
        'val_loss': val_loss,
        'model_name': 'osnet_x1_0',
        'num_classes': NUM_CLASSES,
        'input_size': list(IMAGE_SIZE),
        'pixel_mean': list(MEAN),
        'pixel_std': list(STD),
    }

def save_checkpoint(path, pack, include_training_state=False):
    payload = dict(pack)
    if include_training_state:
        payload['optimizer'] = optimizer.state_dict()
        payload['scheduler'] = scheduler.state_dict()
        payload['scaler'] = scaler.state_dict()
    temp_path = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temp_path)
    os.replace(temp_path, path)

start_epoch, best_map = 1, -1.0
if RESUME and not SMOKE:
    resume_paths = sorted(Path('/kaggle/input').rglob('lastOsnet.pt'))
    if len(resume_paths) > 1:
        raise RuntimeError(f'Hay varios lastOsnet.pt adjuntos: {resume_paths}. Deja solo uno o desactiva RESUME.')
    resume_path = resume_paths[0] if resume_paths else None
    if resume_path is not None:
        state = torch.load(resume_path, map_location='cpu', weights_only=False)
        if state.get('model_name') != 'osnet_x1_0':
            raise RuntimeError(f'Checkpoint no compatible con OSNet x1.0: {resume_path}')
        model.load_state_dict(state['state_dict'])
        if 'optimizer' in state:
            optimizer.load_state_dict(state['optimizer'])
            scheduler.load_state_dict(state['scheduler'])
            scaler.load_state_dict(state['scaler'])
        start_epoch = int(state.get('epoch', 0)) + 1
        best_map = float(state.get('best_val_map', -1.0))
        del state
        print(f'Reanudando desde {resume_path} | epoch anterior={start_epoch-1} | mejor mAP={best_map:.4f}')

# ---------- batch-hard triplet loss ----------
def batch_hard_triplet(embeddings, labels):
    embeddings = F.normalize(embeddings.float(), p=2, dim=1)
    distances = torch.cdist(embeddings, embeddings, p=2)
    same = labels[:, None].eq(labels[None, :])
    eye = torch.eye(len(labels), dtype=torch.bool, device=labels.device)
    positive = same & ~eye
    negative = ~same
    hardest_positive = distances.masked_fill(~positive, float('-inf')).max(dim=1).values
    hardest_negative = distances.masked_fill(~negative, float('inf')).min(dim=1).values
    losses = F.relu(hardest_positive - hardest_negative + MARGIN)
    return losses.mean(), (losses > 0).float().mean()

def get_embeddings(output):
    return output[1] if isinstance(output, (tuple, list)) else output

@torch.no_grad()
def evaluate_val():
    model.eval()
    # --- proxy de val loss con batches P×K deterministas ---
    loss_sum, n = 0.0, 0
    for batch in val_sampler:
        idx = torch.as_tensor(batch, device=device)
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(prep(val_x[idx], False))
        loss, _ = batch_hard_triplet(get_embeddings(out), val_y[idx])
        loss_sum += loss.item()
        n += 1
    val_loss = loss_sum / max(n, 1)
    # --- retrieval sobre todas las muestras fijas de val ---
    feats = []
    for i in range(0, len(val_x), 128):
        with torch.autocast('cuda', dtype=torch.float16):
            out = model(prep(val_x[i:i + 128], False))
        feats.append(F.normalize(get_embeddings(out).float(), p=2, dim=1))
    feats = torch.cat(feats)
    dist = 1.0 - feats @ feats.T
    dist.fill_diagonal_(float('inf'))
    order = dist.argsort(dim=1)[:, :-1]                 # descarta la propia muestra (última, dist=inf)
    matches = val_y[order].eq(val_y[:, None])
    rank1 = matches[:, 0].float().mean().item()
    ranks = torch.arange(1, matches.shape[1] + 1, device=device, dtype=torch.float32)[None, :]
    precision = matches.cumsum(dim=1).float() / ranks
    positives = matches.sum(dim=1).clamp_min(1)
    ap = (precision * matches).sum(dim=1) / positives
    return val_loss, rank1, ap.mean().item()

# ---------- entrenamiento ----------
started = time.time()
for epoch in range(start_epoch, EPOCHS + 1):
    epoch_start = time.time()
    train_sampler.set_epoch(epoch)
    model.train()
    loss_sum, active_sum, steps_done = 0.0, 0.0, 0
    for step, batch in enumerate(train_sampler, start=1):
        idx = np.asarray(batch)
        images = torch.from_numpy(train_x[idx]).to(device, non_blocking=True)
        labels = train_y_t[idx].to(device, non_blocking=True)
        images = prep(images, True)
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast('cuda', dtype=torch.float16):
            output = model(images)
        loss, active = batch_hard_triplet(get_embeddings(output), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item()
        active_sum += active.item()
        steps_done += 1
        if step == 1 or step % 50 == 0:
            print(f'  epoch {epoch} step {step}/{len(train_sampler)} | triplet {loss_sum/steps_done:.4f} | '
                  f'active {active_sum/steps_done:.2f} | {time.time()-epoch_start:.0f}s', flush=True)
        if step % SAVE_EVERY_STEPS == 0:
            save_checkpoint(OUT / 'last.pt', checkpoint_pack(epoch - 1, best_map, 0.0, 0.0), True)
    scheduler.step()

    val_loss, val_rank1, val_map = evaluate_val()
    lr_now = optimizer.param_groups[0]['lr']
    minutes = (time.time() - epoch_start) / 60
    train_loss = loss_sum / max(steps_done, 1)
    print(f'Epoch {epoch}/{EPOCHS} | train triplet {train_loss:.4f} | val triplet {val_loss:.4f} | '
          f'Rank-1 {val_rank1:.4f} | mAP {val_map:.4f} | lr {lr_now:.2e} | {minutes:.1f} min', flush=True)
    with metrics_path.open('a', newline='', encoding='utf-8') as f:
        csv.writer(f).writerow([epoch, train_loss, val_loss, val_rank1, val_map, minutes])

    pack = checkpoint_pack(epoch, max(best_map, val_map), val_rank1, val_loss)
    save_checkpoint(OUT / 'last.pt', pack, True)
    if val_map > best_map:
        best_map = val_map
        pack['best_val_map'] = best_map
        save_checkpoint(OUT / 'best.pt', pack, False)
        print(f'  Nuevo mejor checkpoint: {OUT / "best.pt"} | val mAP={best_map:.4f}', flush=True)
    gc.collect()
    torch.cuda.empty_cache()
    if (time.time() - started) / 3600 >= MAX_HOURS:
        print('Límite horario alcanzado; last.pt queda listo para reanudar.', flush=True)
        break

print(f'Entrenamiento terminado. Mejor val mAP={best_map:.4f}.')
print('Archivos:', sorted(p.name for p in OUT.iterdir()))
# ===============================================================================